# Projeto 01 — Agente Inteligente para Distribuição de Produtos Locais

**Unidade Curricular:** Inteligência Artificial  
**Curso:** Engenharia de Sistemas Informáticos  
**Ano Letivo:** 2026/2027  
**Grupo:** G05

### Elementos do grupo

- David Faria — 31517
- Francisco Campinho — 31079
- Rafael Costa — 31524
- Miguel Marques — 33221

---

Este notebook corresponde ao documento principal do Projeto 01. Nesta fase são apresentados o contexto do problema, os principais resultados da análise exploratória dos dados e a abordagem inicial considerada para a sua modelação.

## 1. Introdução

### 1.1 Contexto e motivação

A distribuição de produtos alimentares locais permite aproximar pequenos produtores dos consumidores, reduzir a dimensão das cadeias de abastecimento e potencialmente diminuir deslocações desnecessárias e desperdício alimentar.

Apesar destas vantagens, a distribuição local apresenta vários desafios logísticos. Os produtos encontram-se distribuídos por diferentes produtores, os clientes podem escolher diferentes locais de entrega e cada operação de recolha e entrega está sujeita a limitações temporais. Para além disso, a capacidade dos veículos e os tempos de deslocação entre os diferentes pontos devem ser considerados durante o planeamento.

Neste projeto é considerada uma plataforma digital que relaciona três intervenientes principais:

- **produtores**, que disponibilizam os produtos;
- **clientes**, que efetuam encomendas e escolhem um local de entrega;
- **centros de recolha**, que funcionam como base operacional dos veículos responsáveis pela distribuição.

Assim, o problema não consiste apenas em encontrar o percurso geograficamente mais curto. Uma rota com uma distância reduzida pode continuar a ser inválida se, por exemplo, um veículo chegar a um produtor fora da sua janela de disponibilidade, entregar um produto depois do limite estabelecido ou transportar uma carga superior à sua capacidade.

### 1.2 Objetivo do projeto

O objetivo do projeto é desenvolver um **agente inteligente de roteamento** capaz de construir rotas de recolha e entrega eficientes e, simultaneamente, respeitar as restrições operacionais definidas no problema.

Para cada item de uma encomenda, o agente deverá considerar:

- o produtor responsável pelo produto e a respetiva localização;
- o local de entrega associado à encomenda;
- a quantidade e o peso do produto;
- a janela temporal durante a qual o produto pode ser recolhido;
- a janela temporal durante a qual o produto deve ser entregue;
- a capacidade disponível do veículo;
- a distância e o tempo de deslocação entre localizações;
- a precedência obrigatória entre recolha e entrega.

Numa fase posterior serão estudadas diferentes técnicas de Inteligência Artificial, nomeadamente pesquisa informada, pesquisa local e resolução através de um Constraint Satisfaction Problem (CSP).

Nesta primeira fase, o trabalho concentra-se principalmente na **compreensão e validação dos dados** e na definição inicial da representação do problema.

### 1.3 Fluxo operacional considerado

De forma simplificada, uma operação de distribuição segue a seguinte sequência:

**Centro de recolha → Produtor → Local de entrega**

No entanto, uma rota real poderá incluir várias recolhas e várias entregas antes do regresso ao centro.

Cada `OrderItem` representa uma necessidade concreta de transporte. Através do `product_id` é possível identificar o produtor onde o produto deve ser recolhido, enquanto através do `order_id` é possível determinar o local onde esse produto deverá ser entregue.

Consequentemente, cada item de encomenda pode ser interpretado como um pedido de **pickup-and-delivery** com duas janelas temporais:

1. uma janela de disponibilidade no produtor;
2. uma janela de entrega no destino.

Esta interpretação será utilizada como base para a modelação posterior do agente.

## 2. Resumo da Análise Exploratória dos Dados

A análise exploratória completa encontra-se no notebook `EDA.ipynb`. Nesta secção são apresentados apenas os resultados mais relevantes para a posterior modelação do problema.

O dataset `tiny` é constituído por:

- **49 localidades**;
- **5 produtores**;
- **23 produtos**;
- **8 clientes**;
- **10 encomendas**;
- **23 itens de encomenda**;
- **1 centro de recolha**;
- **2 veículos**;
- **12 localizações relevantes para o roteamento**;
- **132 ligações origem-destino** na matriz de distâncias.

A análise inicial não revelou valores em falta nem linhas integralmente duplicadas nos ficheiros fornecidos.

Para além destas verificações básicas, foram analisadas as relações entre as entidades, as janelas temporais, a procura em termos de peso, a capacidade dos veículos e a matriz de distâncias e tempos.

### 2.1 Janelas temporais

As janelas temporais constituem uma das restrições mais importantes do problema.

No dataset `tiny`, as janelas de disponibilidade para recolha apresentam uma duração média de aproximadamente **3,87 horas**, com valores entre 2 e 6 horas.

As janelas de entrega apresentam uma duração média de aproximadamente **2,78 horas**, variando entre 1 e 4 horas.

Não foram encontradas janelas em que o instante inicial seja posterior ao instante final. Foram, contudo, identificados **6 itens de encomenda cuja recolha e entrega estão previstas para dias diferentes**.

Este resultado demonstra que a futura representação do estado não deverá assumir que todas as operações associadas a um pedido ocorrem obrigatoriamente no mesmo dia.

### 2.2 Peso das encomendas e capacidade

A partir da quantidade de cada item e do peso unitário do respetivo produto foi estimado o peso associado a cada encomenda.

A encomenda individual mais pesada apresenta aproximadamente **49,90 kg**.

Os dois veículos disponíveis possuem capacidades de:

- **1444,7 kg**;
- **1009,8 kg**.

Nenhuma encomenda ultrapassa individualmente a capacidade do veículo mais pequeno.

Por esse motivo, no dataset `tiny`, a capacidade não aparenta ser a principal fonte de dificuldade. Ainda assim, esta restrição não pode ser eliminada da modelação, uma vez que um veículo poderá transportar simultaneamente produtos pertencentes a várias encomendas e o dataset de maior dimensão poderá apresentar uma situação diferente.

O conjunto completo dos 23 itens corresponde a aproximadamente **248,96 kg** de produto.

Este valor reforça que, no pequeno exemplo disponibilizado, as restrições temporais deverão ter maior influência sobre a construção das rotas do que a capacidade máxima dos veículos.

### 2.3 Rede de localizações

A rede utilizada pelo agente contém **12 localizações**:

- 5 produtores;
- 1 centro de recolha;
- 6 locais de entrega.

Foi ainda observado que dois produtores diferentes possuem exatamente as mesmas coordenadas geográficas. Como consequência, existem ligações entre localizações diferentes com distância e duração iguais a zero.

Esta situação é válida no contexto do dataset e deverá ser tratada naturalmente pelos algoritmos, sem assumir que dois identificadores de localização diferentes têm obrigatoriamente uma distância positiva entre si.

### 2.4 Distâncias e tempos de viagem

Para as 12 localizações existem:

\[
12 \times (12 - 1) = 132
\]

pares origem-destino distintos.

A matriz fornecida contém exatamente estes **132 pares**, sem pares duplicados, distâncias negativas ou tempos de viagem negativos.

Foi também verificada a simetria da matriz: no dataset `tiny`, a distância e a duração entre duas localizações são iguais nos dois sentidos.

A distância média entre pontos é aproximadamente **35,72 km**, enquanto o tempo médio de deslocação é aproximadamente **47,63 minutos**. A maior distância existente é aproximadamente **72,84 km**, correspondendo a um tempo estimado máximo de aproximadamente **97,1 minutos**.

## 3. Definição do Problema

O problema pode ser interpretado como uma variante de um **Vehicle Routing Problem com operações de Pickup and Delivery e Time Windows**.

Pretende-se atribuir pedidos aos veículos disponíveis e determinar a sequência de localizações visitadas de forma a obter rotas válidas e eficientes.

### 3.1 Principais elementos

Consideram-se:

- um conjunto de veículos \(K\);
- um conjunto de localizações \(V\);
- um centro de recolha;
- um conjunto de produtores;
- um conjunto de locais de entrega;
- um conjunto de itens de encomenda;
- uma matriz de distâncias e tempos entre localizações.

Cada item de encomenda origina uma necessidade de recolha num produtor e de entrega no local associado à encomenda.

### 3.2 Principais restrições

Uma solução apenas será considerada válida se respeitar, pelo menos, as seguintes condições:

1. **Precedência** — um produto deve ser recolhido antes de poder ser entregue;
2. **Janela de recolha** — a visita ao produtor deve respeitar o intervalo de disponibilidade do produto;
3. **Janela de entrega** — a entrega deve ocorrer dentro do intervalo definido para o cliente;
4. **Capacidade** — a carga transportada nunca pode ultrapassar a capacidade máxima do veículo;
5. **Deslocação** — o tempo necessário para viajar entre duas localizações deve ser considerado;
6. **Cobertura dos pedidos** — todos os itens considerados na instância deverão ser recolhidos e entregues.

O peso associado a um item pode ser calculado por:

\[
peso_i = quantidade_i \times pesoUnitario_i
\]

À medida que um veículo recolhe produtos, a sua carga aumenta. Quando esses produtos são entregues, a carga correspondente deixa de ocupar capacidade no veículo.

### 3.3 Critério de avaliação

A qualidade de uma solução será avaliada através do custo das rotas construídas.

Numa primeira abordagem, a **distância total percorrida** será utilizada como medida principal de custo:

\[
Custo(R) = \sum_{(i,j)\in R} distancia(i,j)
\]

O tempo de viagem será utilizado principalmente para atualizar o instante atual do veículo e verificar o cumprimento das janelas temporais.

Esta definição poderá ser refinada nas fases seguintes do projeto caso seja necessário considerar simultaneamente outras componentes, como duração total da operação ou tempos de espera.

## 4. Abordagem Inicial

Nesta fase ainda não se pretende resolver completamente o problema. O objetivo é estabelecer uma representação que possa ser reutilizada pelos diferentes algoritmos nas etapas seguintes.

A solução será desenvolvida progressivamente.

### 4.1 Representação da rede

As localizações serão representadas como nós de um grafo.

As deslocações entre duas localizações serão obtidas através de `distance_matrix.csv`, que fornece:

- distância, em quilómetros;
- duração estimada da viagem, em minutos.

Uma rota poderá assim ser representada como uma sequência ordenada de localizações visitadas por um veículo.

### 4.2 Representação de um pedido

Cada `OrderItem` será transformado num pedido de transporte contendo, pelo menos:

- identificador do item;
- identificador da encomenda;
- produtor;
- localização de recolha;
- localização de entrega;
- peso transportado;
- início e fim da janela de recolha;
- início e fim da janela de entrega.

Esta transformação permitirá que os algoritmos trabalhem diretamente com a informação necessária para decidir a próxima ação.

### 4.3 Representação inicial do estado

Um estado do problema deverá permitir conhecer, para cada veículo:

- a localização atual;
- o instante atual;
- a carga atual;
- os pedidos ainda não recolhidos;
- os pedidos já recolhidos e ainda não entregues;
- os pedidos já concluídos;
- a rota realizada;
- o custo acumulado.

Esta representação ainda será refinada durante a fase de implementação.

### 4.4 Próximos passos

O próximo passo será implementar a representação básica das entidades e um mecanismo de verificação de rotas.

Antes de aplicar qualquer algoritmo de pesquisa, serão testadas manualmente pequenas sequências de recolha e entrega de forma a confirmar que:

- os tempos são atualizados corretamente;
- as janelas temporais são respeitadas;
- a carga do veículo é atualizada após cada recolha e entrega;
- uma entrega nunca ocorre antes da respetiva recolha;
- o custo total da rota é calculado corretamente.

Só depois desta validação serão aplicados e comparados os diferentes métodos previstos no projeto:

- Greedy Best-First Search;
- A*;
- Hill-Climbing;
- Constraint Satisfaction Problem (CSP).